# 04. Mamba Selective SSM & Mamba2 SSD

- **Category:** 05_architectures / 03_recurrent_ssm_and_memory
- **Lineage & Citations:** Albert Gu & Tri Dao, 2023 (*Mamba: Linear-Time Sequence Modeling with Selective State Spaces*); Tri Dao & Albert Gu, 2024 (*Transformers are SSMs: Mamba-2*).
- **Purpose:** Selective state space models make recurrence parameters input-dependent ($\Delta_t, B_t, C_t$), enabling associative recall and linear-time sequence modeling with zero KV cache bloat.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Structured State Space Models S4 and LRU](03_structured_state_space_models_s4_and_lru.ipynb) $\to$ `🔗 Downstream Syntheses:` [Linear Attention RWKV RetNet Gla Deltanet and TTT](05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **Dense Cross-Attention Needs:** For multimodal or encoder-decoder cross-attention tasks, standard attention remains superior.
> - **Sub-100 Token Tasks:** For ultra-short sequences, simple feed-forward MLPs avoid selective kernel launch overhead.

## Mathematical Foundations & Governing Equations

### Selective State Space Discretization (Mamba-1)
Continuous parameters are discretized dynamically at each step:
$$
B_t = x_t W_B^\top, \quad C_t = x_t W_C^\top, \quad \Delta_t = \text{softplus}(x_t W_\Delta^\top + b_\Delta)
$$
$$
\bar{A}_t = \exp(\Delta_t \odot A), \quad \bar{B}_t = \Delta_t \otimes B_t
$$
Recurrent state update:
$$
h_t = \bar{A}_t \odot h_{t-1} + \bar{B}_t \odot x_t, \quad y_t = \sum_n (h_{t, \cdot, n} \odot C_{t, \cdot, n}) + x_t \odot D
$$

### Structured State Space Duality (Mamba-2 SSD)
Unrolling the selective recurrence is dual to 1-semiseparable matrix multiplication:
$$
Y = (L \odot (C B^\top)) X, \quad L_{i, j} = \exp\left(\sum_{k=j+1}^i a_k\right) \quad (i \ge j)
$$

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/03_recurrent_ssm_and_memory/04_mamba_selective_ssm_and_mamba2_ssd.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

x_seq = weights["x_seq"]
print(f"Loaded {len(weights)} golden tensors on {env.device}, input shape: {list(x_seq.shape)}")

In [ ]:
def mamba_selective_forward(x_seq, a_log, w_b, w_c, w_delta, b_delta, d_skip):
    B, T, D_model = x_seq.shape
    D_state = a_log.shape[1]
    a_real = -torch.exp(a_log)
    h_t = torch.zeros(B, D_model, D_state, device=x_seq.device, dtype=x_seq.dtype)
    outs = []
    for t in range(T):
        xt = x_seq[:, t, :]
        b_t = xt @ w_b.t()
        c_t = xt @ w_c.t()
        delta_t = F.softplus(xt @ w_delta.t() + b_delta)
        a_bar = torch.exp(delta_t.unsqueeze(-1) * a_real.unsqueeze(0))
        b_bar = delta_t.unsqueeze(-1) * b_t.unsqueeze(1)
        h_t = a_bar * h_t + b_bar * xt.unsqueeze(-1)
        y_t = (h_t * c_t.unsqueeze(1)).sum(dim=-1) + xt * d_skip
        outs.append(y_t.unsqueeze(1))
    return torch.cat(outs, dim=1), h_t

def mamba2_ssd_mask(delta_seq, a_scalar):
    decay = -F.softplus(delta_seq) * torch.abs(a_scalar)
    cumsum_decay = torch.cumsum(decay, dim=-1)
    diff = cumsum_decay.unsqueeze(2) - cumsum_decay.unsqueeze(1)
    mask = torch.tril(torch.ones(delta_seq.shape[1], delta_seq.shape[1], device=delta_seq.device, dtype=torch.bool))
    return torch.where(mask.unsqueeze(0), torch.exp(diff), torch.zeros_like(diff))

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
seq_out, final_h = mamba_selective_forward(
    x_seq, weights["a_log"], weights["w_b"], weights["w_c"],
    weights["w_delta"], weights["b_delta"], weights["d_skip"]
)
err_seq = float(torch.max(torch.abs(seq_out - weights["seq_out"])))
err_h = float(torch.max(torch.abs(final_h - weights["final_h"])))
assert err_seq < 1e-6, f"Mamba sequence error: {err_seq:.2e}"
assert err_h < 1e-6, f"Mamba final_h error: {err_h:.2e}"

l_matrix = mamba2_ssd_mask(weights["delta_scalar"], weights["a_scalar"])
err_l = float(torch.max(torch.abs(l_matrix - weights["l_matrix"])))
assert err_l < 1e-6, f"SSD decay matrix error: {err_l:.2e}"
print(f"Parity Verified: Mamba Seq={err_seq:.2e}, Final H={err_h:.2e}, SSD L-matrix={err_l:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_mamba = benchmark_fn(lambda: mamba_selective_forward(
    x_seq, weights["a_log"], weights["w_b"], weights["w_c"],
    weights["w_delta"], weights["b_delta"], weights["d_skip"]
), iters=50, name="Mamba Selective SSM")
b_ssd = benchmark_fn(lambda: mamba2_ssd_mask(weights["delta_scalar"], weights["a_scalar"]), iters=50, name="Mamba-2 SSD Mask")
print(format_benchmark_table([b_mamba, b_ssd]))

# Visualization: Selective Trajectory & Causal Decay Mask
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].imshow(seq_out[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[0].set_title("Mamba Output Trajectory [T, D]")
ax[1].imshow(l_matrix[0].detach().numpy(), cmap="magma", aspect="auto")
ax[1].set_title("SSD 1-Semiseparable Mask L")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Unbounded Positive State Dynamics (Explosion)
a_unstable = torch.tensor([[1.5]])  # A > 0 causes exponential blowup instead of contractive decay
h_exploding = torch.tensor([1.0])
delta = 0.5
for _ in range(15):
    a_bar_unstable = torch.exp(delta * a_unstable)
    h_exploding = a_bar_unstable * h_exploding

print(f"Unstable state value after 15 steps: {h_exploding.item():.2e}")
assert h_exploding.item() > 1e4, "Unstable A must produce explosive magnitude!"
print("Demonstrated requirement of negative log-parameterized A for contractive Hurwitz stability.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Input-Dependent Selectivity:** Selectively filtering inputs ($\Delta_t, B_t, C_t$) enables associative recall while preserving $\mathcal{O}(T)$ linear complexity.
2. **SSD Duality:** Mamba-2 unifies selective SSMs with attention via 1-semiseparable matrix multiplications, utilizing GPU tensor cores at full speed.
3. **Eliminates KV Cache:** Replaces unbounded auto-regressive memory with a fixed state footprint $\mathcal{O}(D \cdot N)$.

---
### Cross-Domain Generalisation Matrix
- **Long-Context Genomics:** Efficient processing of million-token DNA sequences without quadratic memory blowup.
- **Real-Time Audio Synthesis:** Constant latency generation for streaming speech codecs.
- **Irregular Time-Series:** Dynamically adjustable $\Delta_t$ natively models variable time gaps.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_mamba_selective_ssm_and_mamba2_ssd.md](../../../markdown/05_architectures/03_recurrent_ssm_and_memory/04_mamba_selective_ssm_and_mamba2_ssd.md)
- **PyTorch 2.x Module:** [04_mamba_selective_ssm_and_mamba2_ssd.py](../../../pytorch2/05_architectures/03_recurrent_ssm_and_memory/04_mamba_selective_ssm_and_mamba2_ssd.py)
- **Functional JAX Module:** [04_mamba_selective_ssm_and_mamba2_ssd.py](../../../jax/05_architectures/03_recurrent_ssm_and_memory/04_mamba_selective_ssm_and_mamba2_ssd.py)
- **Candle Rust Module:** [04_mamba_selective_ssm_and_mamba2_ssd.rs](../../../candle/05_architectures/03_recurrent_ssm_and_memory/04_mamba_selective_ssm_and_mamba2_ssd.rs)
- **Hardware Kernel:** [04_mamba_selective_ssm_and_mamba2_ssd.rs](../../../kernels/05_architectures/03_recurrent_ssm_and_memory/04_mamba_selective_ssm_and_mamba2_ssd.rs)
- **Universal Golden SafeTensors:** [04_mamba_selective_ssm_and_mamba2_ssd.safetensors](../../../weights/05_architectures/03_recurrent_ssm_and_memory/04_mamba_selective_ssm_and_mamba2_ssd.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 03. Structured State Space Models S4 and LRU](03_structured_state_space_models_s4_and_lru.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 05. Linear Attention RWKV RetNet Gla Deltanet and TTT](05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.ipynb) &rarr;